In [2]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [3]:
tf.__version__

'2.19.1'

In [7]:
# load and preprocess the dataset
data = load_iris()
X = data.data
y = data.target

# split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
# define the model-building function for Keras Tuner
def build_model(hp):
    model = keras.Sequential()

    # tune the number of units in the first hidden layer
    model.add(keras.layers.Dense(
        units=hp.Int("units", min_value=8, max_value=64, step=8),
        activation="relu",
        input_shape=(X_train.shape[1],)
    ))

    # add dropout layer and tune dropout rate
    model.add(keras.layers.Dropout(
        rate=hp.Float("dropout", min_value=0.0, max_value=0.5, step=0.1)
    ))

    # add output layer
    model.add(keras.layers.Dense(3, activation="softmax"))

    # tune the optimizer type
    optimizer = hp.Choice("optimizer", values=["adam", "sgd", "rmsprop"])

    # tune the learning rate
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")

    if optimizer == "adam":
        opt = keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer == "sgd":
        opt = keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        opt = keras.optimizers.RMSprop(learning_rate=learning_rate)

    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# initialize Keras Tuner (Random Search)
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=10, # number of hyperparameter combination to try
    executions_per_trial=2, # number of models to train per trial
    directory="tuner",
    project_name="keras-tuner-iris-dataset"
)

In [17]:
# search for the best hyperparameters
tuner.search(X_train, y_train, epochs=20, validation_data=(X_test, y_test))

# get the best model
# best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = build_model(best_hps) # build the fresh model with best hyperparameter

# train the model (optional, retraining for best hyperparameter)
best_model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test), verbose=0)

In [18]:
# evaluate the best model
test_loss, test_accuracy = best_model.evaluate(X_test, y_test)
print(f"\nBest Model Test Accuracy: {test_accuracy:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.9000 - loss: 0.2903

Best Model Test Accuracy: 0.9000


In [19]:
# print the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Hyperparameters")
print(f" - Units of hidden layer: {best_hps.get('units')}")
print(f" - Dropout rate: {best_hps.get('dropout')}")
print(f" - Optimizer: {best_hps.get('optimizer')}")
print(f" - Learning rate: {best_hps.get('learning_rate')}")

Best Hyperparameters
 - Units of hidden layer: 48
 - Dropout rate: 0.2
 - Optimizer: adam
 - Learning rate: 0.0018242584647859385
